In [1]:
import numpy as np
from scipy.stats import binom
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# --- ENVIRONMENT & SYSTEM CONFIGURATION ---
NUM_CHANNELS = 5          # Reduced slightly for faster quantum simulation runtime
TIME_STEPS = 4
PU_OCCUPANCY_PROB = 0.4   # Probability a Primary User occupies a channel
N_WINDOW = 40             # Number of entangled photon pairs (quantum probes) sent per sensing cycle
TARGET_PFA = 0.05         # Maximum acceptable False Alarm rate
SU_SENSING_SNR_DB = 4.0   # The Signal-to-Noise Ratio of our quantum probe if PU is present

# Initialize the Quantum Simulator
_BACKEND = AerSimulator()

def snr_db_to_ploss(snr_db, floor=0.02, ceiling=0.98):
    """Maps the operating SNR to a depolarizing channel error probability."""
    gamma = 10 ** (snr_db / 10.0)
    p_loss = 1.0 / (1.0 + gamma)
    return float(np.clip(p_loss, floor, ceiling))

def _quantum_circuit():
    """
    Builds a 2-qubit circuit implementing the Quantum Illumination protocol.
    Qubit 0: Idler photon (retained securely at the receiver).
    Qubit 1: Signal photon (transmitted into the environment to probe for PU).
    """
    qc = QuantumCircuit(2, 2)
    
    # 1. State Preparation: Generate an entangled Bell State (|Φ+>)
    qc.h(0)
    qc.cx(0, 1)
    
    # 2. Channel Interaction: Signal photon (Qubit 1) interacts with the environment
    qc.id(1) 
    
    # 3. Quantum Receiver: Re-combine and decode via joint entangled measurement
    qc.cx(0, 1)
    qc.h(0)
    
    # Measure both photons
    qc.measure([0, 1], [0, 1])
    return qc

def calculate_decision_threshold(p0, n_window, target_pfa):
    """Computes the optimal statistical hit threshold (tau) using Neyman-Pearson logic."""
    tau = n_window + 1
    for t in range(n_window + 1):
        pfa_t = 1.0 if t == 0 else binom.sf(t - 1, n_window, p0)
        if pfa_t <= target_pfa:
            tau = t
            break
    return tau

def get_channel_truth():
    """Generates ground truth: 1 if PU is active (Occupied), 0 if Idle."""
    return np.random.choice([0, 1], size=NUM_CHANNELS, p=[1-PU_OCCUPANCY_PROB, PU_OCCUPANCY_PROB])

def run_dynamic_quantum_sensing():
    # --- STEP 1: CALIBRATION ---
    # We calibrate the receiver by observing the circuit's behavior under pure noise (PU Absent)
    qc = _quantum_circuit()
    qc_compiled = transpile(qc, basis_gates=["h", "cx", "id"], optimization_level=0)
    
    nm_h0 = NoiseModel()
    nm_h0.add_quantum_error(depolarizing_error(1.0, 1), "id", [1]) # 100% loss/noise
    cal_job = _BACKEND.run(qc_compiled, shots=2000, noise_model=nm_h0)
    cal_counts = cal_job.result().get_counts()
    
    # Baseline probability of successfully decoding the state '00' when no target is present
    p0 = cal_counts.get("00", 0) / 2000
    
    # Compute the rigorous hit threshold required to guarantee our target Pfa constraint
    tau = calculate_decision_threshold(p0, N_WINDOW, TARGET_PFA)
    
    print("\n" + "=" * 70)
    print("                DYNAMIC QUANTUM ILLUMINATION SENSING RUN")
    print("=" * 70)
    print(f"Calibration Metrics -> Baseline p0: {p0:.4f} | Optimal Hit Threshold (tau): {tau}/{N_WINDOW}")
    print("-" * 70)
    print(f"{'Time':<6} | {'Channel':<8} | {'Truth':<6} | {'QI Hits':<8} | {'Decision':<10} | {'Status'}")
    print("-" * 70)
    
    # --- STEP 2: DYNAMIC SIMULATION LOOP ---
    p_loss = snr_db_to_ploss(SU_SENSING_SNR_DB)
    
    for t in range(TIME_STEPS):
        ground_truth = get_channel_truth()
        
        for ch in range(NUM_CHANNELS):
            # If PU is active, the channel reflects the signal with SNR-dependent attenuation (p_loss)
            # If PU is absent, the signal photon is completely lost to background thermal noise (1.0)
            p_dep = p_loss if ground_truth[ch] == 1 else 1.0
            
            # Setup the real-time physical noise model for this specific channel slot
            nm = NoiseModel()
            nm.add_quantum_error(depolarizing_error(p_dep, 1), "id", [1])
            
            # Fire the window batch of entangled photon pairs through the quantum simulator
            job = _BACKEND.run(qc_compiled, shots=N_WINDOW, noise_model=nm)
            counts = job.result().get_counts()
            
            # Total instances where the idler and signal photons successfully collapsed back into '00'
            hits = counts.get("00", 0)
            
            # Cognitive Radio Decision Engine
            observed = 1 if hits >= tau else 0
            
            # Performance tracking evaluation
            if ground_truth[ch] == 1:
                status = "Correct Detection" if observed == 1 else "Missed Detection (Collision)"
            else:
                status = "False Alarm" if observed == 1 else "Correct Idle"
            
            decision_str = "Occupied" if observed == 1 else "Idle"
            print(f"{t:<6} | {ch:<8} | {ground_truth[ch]:<6} | {hits:<8} | {decision_str:<10} | {status}")
        print("-" * 70)

if __name__ == "__main__":
    run_dynamic_quantum_sensing()


                DYNAMIC QUANTUM ILLUMINATION SENSING RUN
Calibration Metrics -> Baseline p0: 0.2475 | Optimal Hit Threshold (tau): 16/40
----------------------------------------------------------------------
Time   | Channel  | Truth  | QI Hits  | Decision   | Status
----------------------------------------------------------------------
0      | 0        | 0      | 11       | Idle       | Correct Idle
0      | 1        | 0      | 9        | Idle       | Correct Idle
0      | 2        | 0      | 11       | Idle       | Correct Idle
0      | 3        | 0      | 9        | Idle       | Correct Idle
0      | 4        | 1      | 32       | Occupied   | Correct Detection
----------------------------------------------------------------------
1      | 0        | 0      | 10       | Idle       | Correct Idle
1      | 1        | 0      | 10       | Idle       | Correct Idle
1      | 2        | 1      | 31       | Occupied   | Correct Detection
1      | 3        | 0      | 11       | Idle       